In [ ]:
import pandas as pd
df = pd.read_csv('copy1.csv', sep=',')


In [27]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)
for i, col in enumerate(df.columns):
    print(i, repr(col))
label_counts = df['label'].value_counts()
print(label_counts)


0 'destination port'
1 'flow duration'
2 'total fwd packets'
3 'total backward packets'
4 'total length of fwd packets'
5 'total length of bwd packets'
6 'fwd packet length max'
7 'fwd packet length min'
8 'fwd packet length mean'
9 'fwd packet length std'
10 'bwd packet length max'
11 'bwd packet length min'
12 'bwd packet length mean'
13 'bwd packet length std'
14 'flow bytes/s'
15 'flow packets/s'
16 'flow iat mean'
17 'flow iat std'
18 'flow iat max'
19 'flow iat min'
20 'fwd iat total'
21 'fwd iat mean'
22 'fwd iat std'
23 'fwd iat max'
24 'fwd iat min'
25 'bwd iat total'
26 'bwd iat mean'
27 'bwd iat std'
28 'bwd iat max'
29 'bwd iat min'
30 'fwd psh flags'
31 'bwd psh flags'
32 'fwd urg flags'
33 'bwd urg flags'
34 'fwd header length'
35 'bwd header length'
36 'fwd packets/s'
37 'bwd packets/s'
38 'min packet length'
39 'max packet length'
40 'packet length mean'
41 'packet length std'
42 'packet length variance'
43 'fin flag count'
44 'syn flag count'
45 'rst flag count'
46 'ps

In [31]:
# 1. Tách các dataframe riêng biệt theo nhãn
df_benign = df[df['label'] == 'BENIGN']
df_ftp = df[df['label'] == 'FTP-Patator']
df_ssh = df[df['label'] == 'SSH-Patator']

# 2. Tính tổng số lượng tấn công
total_attack_count = len(df_ftp) + len(df_ssh)
print(f"\nTổng số lượng mẫu tấn công (FTP + SSH): {total_attack_count}")



Tổng số lượng mẫu tấn công (FTP + SSH): 13835


In [32]:
# 3. Kỹ thuật UNDERSAMPLING (Giảm mẫu Benign)
# Chiến thuật: Lấy số lượng Benign = Tổng số lượng Attack (Tỷ lệ 1:1)
# Hoặc nhân hệ số k (ví dụ 1.5) nếu muốn Benign nhiều hơn một chút cho thực tế
# Ở đây tôi chọn tỷ lệ 1:1 để Model học tốt nhất sự khác biệt
df_benign_sampled = df_benign.sample(n=total_attack_count, random_state=42)
print(f"Số lượng Benign sau khi lấy mẫu ngẫu nhiên: {len(df_benign_sampled)}")

Số lượng Benign sau khi lấy mẫu ngẫu nhiên: 13835


In [33]:
# 4. Gộp lại thành Dataset hoàn chỉnh (Concatenate)
df_balanced = pd.concat([df_benign_sampled, df_ftp, df_ssh])

In [36]:
# 5. Xáo trộn dữ liệu (Shuffle) - Bước cực quan trọng
# Nếu không xáo trộn, model sẽ học theo thứ tự (VD: học hết Benign rồi mới tới Attack) -> Dễ bị sai
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print("\n--- THỐNG KÊ SAU KHI CÂN BẰNG (READY TO TRAIN) ---")
print(df_balanced['label'].value_counts())


--- THỐNG KÊ SAU KHI CÂN BẰNG (READY TO TRAIN) ---
label
BENIGN         13835
FTP-Patator     7938
SSH-Patator     5897
Name: count, dtype: int64


In [37]:
# 6. Kiểm tra lại kích thước
print(f"\nKích thước dataset cuối cùng: {df_balanced.shape}")


Kích thước dataset cuối cùng: (27670, 79)


In [38]:
# Lưu ra file mới để dùng cho việc training
df_balanced.to_csv('tuesday_balanced_label_training_1.csv', index=False)
print("Đã lưu file 'tuesday_balanced_label_training_1.csv' thành công!")

Đã lưu file 'tuesday_balanced_label_training_1.csv' thành công!
